



Gus, la propuesta es correcta, pero haría **dos ajustes importantes**:

1. Separaría el diseño experimental de la generación física de ventanas.
2. No ejecutaría inicialmente todos los modelos × todas las ventanas × todos los folds × múltiples hiperparámetros, porque el número de entrenamientos crecería demasiado y aumentaría el riesgo de seleccionar resultados por azar.

Los modelos complejos tienen mayor capacidad para aprender relaciones no lineales, pero también mayor riesgo de sobreajuste, especialmente con datos financieros de baja relación señal-ruido. fileciteturn0file0 fileciteturn0file6

## Objetivo del Stage_07

Evaluar modelos neuronales capaces de aprender relaciones temporales para:

```text
Target principal:
opc_p50_h60_tp15_sl10

Contexto inicial:
all_regimes

Features principales:
OPC_reduced_no_level
```

Mantendría `OPC_reduced_level` como experimento secundario. Las features de nivel mostraron mucha información mutua, pero podrían estar capturando año, contrato o nivel nominal del MNQ y generalizar peor.

## Estructura recomendada

```text
Stage_07/
│
├── S07_00_experimental_design.ipynb
├── S07_01_sequence_dataset.ipynb
├── S07_02_mlp_classifier.ipynb
├── S07_03_cnn1d_classifier.ipynb
├── S07_04_lstm_classifier.ipynb
├── S07_05_gru_classifier.ipynb
├── S07_06_tcn_classifier.ipynb
└── S07_07_model_comparison.ipynb
```

### `S07_00_experimental_design.ipynb`

Debe definir:

- objetivo y alcance del stage;
- target y codificación de clases;
- datasets utilizados;
- folds walk-forward;
- ventanas temporales candidatas;
- modelos;
- métricas;
- protocolo de entrenamiento;
- estructura estándar de las notebooks;
- criterios para seleccionar modelos;
- estructura de archivos de salida.

Aquí también dejaría la configuración central:

```python
TARGET_NAME = "opc_p50_h60_tp15_sl10"

LOOKBACK_WINDOWS = [30, 60, 90]

WALK_FORWARD_FOLDS = ["WF_01", "WF_02", "WF_03"]

RANDOM_SEEDS = [42]

PRIMARY_METRIC = "macro_f1"
```

### `S07_01_sequence_dataset.ipynb`

Aquí generaría las ventanas reutilizables por todos los modelos.

No conviene guardar tres copias completas de matrices tridimensionales. Es preferible guardar:

```text
X_base
y
timestamps
trading_date
fold
window_end_indices
```

Y construir cada secuencia desde un `Dataset` o generador durante el entrenamiento.

Esto evita archivos enormes y permite usar las mismas observaciones en todos los modelos.

## Representación 2D y 3D

La representación canónica debería ser tridimensional:

```text
X_sequence.shape =
(n_samples, lookback_minutes, n_features)
```

Ejemplo:

```text
(500000, 60, 20)
```

Los modelos utilizarían esa misma información de la siguiente manera:

| Modelo | Entrada |
|---|---|
| MLP | `(samples, lookback × features)` |
| CNN 1D | `(samples, lookback, features)` |
| LSTM | `(samples, lookback, features)` |
| GRU | `(samples, lookback, features)` |
| TCN | `(samples, lookback, features)` |

De esta manera, todos reciben exactamente el mismo historial. Solamente cambia la forma en que lo procesan.

Las CNN pueden aprovechar patrones locales dentro de una serie temporal regular. fileciteturn0file10 Las LSTM y GRU incorporan memoria secuencial y están diseñadas para relaciones temporales entre observaciones. fileciteturn0file19

## Reglas obligatorias para las ventanas

Las ventanas deben:

- terminar en el instante `t`;
- contener únicamente información disponible hasta `t`;
- no cruzar días de trading;
- no cruzar discontinuidades de contrato;
- conservar exactamente el target correspondiente a `t`;
- eliminar muestras sin suficiente historia;
- mantener los mismos timestamps para todos los modelos;
- usar `float32` para reducir memoria;
- ajustar cualquier escalador exclusivamente con el train de cada fold.

## Walk-forward

Mantendría exactamente:

```text
WF_01
Train: 2020–2021
Validation: 2022

WF_02
Train: 2020–2022
Validation: 2023

WF_03
Train: 2020–2023
Validation: 2024
```

No lo llamaría K-fold tradicional, sino **walk-forward expanding-window**, porque no existe partición aleatoria ni intercambio temporal entre folds.

Para redes neuronales agregaría dentro de cada train una división interna:

```text
Outer train
├── model_train
└── early_stopping_validation

Outer validation
└── evaluación final del fold
```

La validación anual del walk-forward no debería utilizarse directamente para decidir cuándo detener el entrenamiento, porque eso introduce optimización sobre el conjunto que posteriormente reportaremos como fuera de muestra.

## Modelos propuestos

Orden recomendado:

1. **MLP**  
   Controla si una red neuronal simple mejora los modelos tabulares.

2. **CNN 1D**  
   Busca patrones locales y combinaciones temporales.

3. **LSTM**  
   Modela dependencias secuenciales de mayor alcance.

4. **GRU**  
   Alternativa más simple y normalmente más rápida que LSTM.

5. **TCN**  
   CNN causal con convoluciones dilatadas; suele ser una alternativa sólida para series temporales.

No incorporaría todavía:

- Transformer;
- CNN 2D complejas;
- autoencoders;
- GAN;
- reinforcement learning.

Primero debemos demostrar que existe señal estable con arquitecturas más controlables. El libro también plantea que la complejidad del modelo debe corresponder con la cantidad y calidad de información disponible. fileciteturn0file6

## Métricas estándar

### Métricas principales

```text
Macro F1
Balanced Accuracy
Multiclass Log Loss
```

`Macro F1` debería ser la métrica principal porque asigna la misma importancia a cada clase, independientemente de su frecuencia.

### Diagnóstico por clase

```text
Precision por clase
Recall por clase
F1 por clase
Support por clase
Confusion matrix
```

### Evaluación probabilística

```text
Log Loss
Multiclass Brier Score
Distribución de confianza
Calibración de probabilidades
```

### Métricas operativas preliminares

```text
Predicted trade rate
Predicted NO_TRADE rate
Precision de señales LONG
Precision de señales SHORT
TP rate entre señales operables
Resultados por régimen
Resultados por año
Resultados por fold
```

El resultado económico completo debe quedar para una etapa posterior, porque la evaluación final necesita reglas de entrada, filtrado por probabilidad, costos, slippage y reconstrucción correcta del resultado del trade. El flujo ML4T separa la generación de señales de la simulación y evaluación de la estrategia. fileciteturn0file16

## Estructura estándar de cada notebook de modelo

Todas deberían contener exactamente:

```text
1. Objetivo del modelo
2. Configuración
3. Carga del dataset secuencial
4. Validación de ventanas y folds
5. Arquitectura del modelo
6. Función de entrenamiento
7. Ejecución WF_01
8. Ejecución WF_02
9. Ejecución WF_03
10. Métricas por fold
11. Curvas de entrenamiento
12. Matrices de confusión
13. Predicciones y probabilidades
14. Agregación walk-forward
15. Observaciones
16. Guardado de resultados
```

## Estrategia de ejecución

Para no multiplicar innecesariamente los experimentos:

### Fase A — comparación inicial

```text
Todos los modelos
Lookback = 60 minutos
Tres folds walk-forward
Una configuración conservadora
```

### Fase B — sensibilidad temporal

```text
Los dos mejores modelos
Lookbacks = 30, 60 y 90 minutos
```

### Fase C — optimización

```text
Ajuste limitado de hiperparámetros
Calibración de probabilidades
Selección de umbrales operativos
```

Mi opinión es que esta estructura es más rigurosa que entrenar inmediatamente todos los modelos con todas las ventanas. Mantiene una comparación justa, controla el costo computacional y reduce el riesgo de terminar seleccionando el mejor resultado por simple repetición de experimentos.

# **ESTRUCTURA STAGE 07**

Gus, la notebook `S07_00_experimental_design.ipynb` será el **documento maestro del Stage_07**.

Su función es dejar definido, antes de entrenar nada:

* qué problema queremos resolver;
* qué dataset utilizaremos;
* qué modelos compararemos;
* cómo se construirán las ventanas temporales;
* cómo se entrenarán los modelos;
* cómo se evaluarán;
* qué archivos deberá producir cada notebook.

Piense en ella como el **plano de construcción**. Todavía no construimos los modelos; dejamos todas las reglas establecidas para que cada notebook posterior trabaje exactamente de la misma forma.

## Qué no haremos aquí

En esta notebook no deberíamos:

* entrenar modelos;
* optimizar hiperparámetros;
* generar predicciones;
* comparar resultados finales;
* construir físicamente millones de ventanas 3D.

La generación pesada de ventanas quedará en:

```text
S07_01_sequence_dataset.ipynb
```

En `S07_00` solamente definiremos cómo deben construirse y validaremos que la configuración sea coherente.

---

# Objetivo general de la notebook

La idea principal será:

```text
Definir un protocolo experimental único para evaluar modelos neuronales
sobre el target OPC, utilizando ventanas temporales causales y validación
walk-forward.
```

Esto significa que todos los modelos recibirán:

* el mismo target;
* las mismas features;
* las mismas observaciones;
* las mismas ventanas;
* los mismos folds;
* las mismas métricas.

Así, si CNN, LSTM o GRU obtienen resultados distintos, podremos atribuir la diferencia al modelo y no a cambios en los datos.

---

# Estructura propuesta

## 1. Objetivo y alcance del Stage_07

Aquí explicaremos qué estudiaremos.

Por ejemplo:

```text
El Stage_07 tiene como objetivo evaluar modelos de aprendizaje profundo
capaces de utilizar la estructura temporal de los datos intradiarios del MNQ
para predecir el target operativo OPC.
```

También aclararemos qué queda fuera:

```text
Este stage no realizará todavía el backtesting económico definitivo.
Su objetivo es medir capacidad predictiva fuera de muestra.
```

---

## 2. Punto de partida del Stage_06

Resumiremos brevemente qué recibimos del stage anterior:

* datasets OPC validados;
* features revisadas;
* ausencia de leakage;
* folds walk-forward definidos;
* análisis de Mutual Information;
* selección preliminar de features.

No repetiremos todo el Stage_06. Solo dejaremos documentado desde dónde comenzamos.

---

## 3. Definición del problema predictivo

Aquí definiremos exactamente qué queremos predecir.

```python
TARGET_NAME = "opc_p50_h60_tp15_sl10"
```

También documentaremos las clases OPC:

```text
0 = LONG_TP
1 = LONG_SL
2 = SHORT_TP
3 = SHORT_SL
4 = NO_TRADE
```

Hay que verificar la codificación real del dataset antes de fijarla definitivamente.

El problema será:

```text
Clasificación multiclase de cinco categorías.
```

---

## 4. Dataset experimental principal

Inicialmente propondría:

```python
DATASET_VERSION = "OPC_reduced_no_level"
REGIME_SCOPE = "all_regimes"
```

Esto significa:

* target OPC;
* features reducidas;
* sin features de nivel nominal del precio;
* todos los regímenes intradiarios.

Luego podremos agregar experimentos secundarios con:

```text
OPC_reduced_level
regime_3
otras configuraciones OPC
```

Pero no conviene comenzar mezclando todo.

---

## 5. Definición de las ventanas temporales

Aquí definiremos qué significa una ventana.

Para una ventana de 60 minutos y una observación en el tiempo `t`:

```text
La entrada del modelo contiene las features desde t-59 hasta t.
El target corresponde al instante t.
```

Forma de la matriz:

```text
X.shape = (
    número_de_muestras,
    longitud_de_ventana,
    número_de_features
)
```

Por ejemplo:

```text
X.shape = (500000, 60, 18)
```

Las ventanas candidatas serán:

```python
LOOKBACK_WINDOWS = [30, 60, 90]
```

Pero la primera comparación utilizará:

```python
PRIMARY_LOOKBACK = 60
```

### Reglas de construcción

Las ventanas deberán:

* usar únicamente información pasada y presente;
* finalizar en `t`;
* no utilizar información posterior a `t`;
* no cruzar días;
* no cruzar huecos temporales;
* no cruzar cambios de contrato;
* contener exactamente el número esperado de minutos;
* eliminar observaciones sin historia suficiente.

---

## 6. Esquema walk-forward

Mantendremos los folds del Stage_06:

```text
WF_01
Train: 2020–2021
Validation: 2022

WF_02
Train: 2020–2022
Validation: 2023

WF_03
Train: 2020–2023
Validation: 2024
```

Pero aparecerá una división adicional dentro del train:

```text
Train del fold
├── entrenamiento real del modelo
└── validación interna para early stopping

Validation del fold
└── evaluación fuera de muestra
```

Esto es importante porque el año de validación walk-forward no debe utilizarse para decidir cuándo detener el entrenamiento.

---

## 7. Modelos que se evaluarán

Dejaremos definido el catálogo inicial:

```text
MLP
CNN 1D
LSTM
GRU
TCN
```

### Función de cada modelo

* **MLP:** baseline neuronal sin arquitectura temporal especializada.
* **CNN 1D:** busca patrones locales dentro de la ventana.
* **LSTM:** aprende dependencias temporales mediante memoria.
* **GRU:** similar a LSTM, pero más simple y rápida.
* **TCN:** utiliza convoluciones causales y dilatadas.

No entrenaremos estos modelos aquí. Solamente documentaremos qué modelos tendrán su propia notebook.

---

## 8. Protocolo estándar de entrenamiento

Todos los modelos deberán utilizar las mismas reglas:

```text
Una semilla inicial
Mismos folds
Mismas ventanas
Mismas features
Misma codificación del target
Mismas métricas
Early stopping
Guardado del mejor modelo
```

Configuración inicial:

```python
RANDOM_SEED = 42
MAX_EPOCHS = 50
EARLY_STOPPING_PATIENCE = 5
BATCH_SIZE = 512
```

Estos valores serán configuraciones iniciales, no necesariamente definitivas.

---

## 9. Métricas de evaluación

La métrica principal será:

```python
PRIMARY_METRIC = "macro_f1"
```

Porque OPC tiene varias clases y probablemente no estén balanceadas.

Métricas generales:

```text
Macro F1
Balanced Accuracy
Log Loss
Accuracy
```

Métricas por clase:

```text
Precision
Recall
F1-score
Support
```

Diagnósticos:

```text
Matriz de confusión
Distribución de predicciones
Probabilidades predichas
Resultados por fold
Resultados por año
Resultados por régimen
```

No utilizaremos solamente `accuracy`, porque un modelo podría obtener una buena accuracy prediciendo excesivamente la clase más frecuente.

---

## 10. Criterios de comparación

Un modelo no será considerado mejor únicamente por tener el mayor resultado en un fold.

Buscaremos:

```text
Buen Macro F1 promedio
Baja variabilidad entre folds
Resultados razonables en todas las clases
Ausencia de colapso hacia NO_TRADE
Probabilidades útiles
Estabilidad temporal
```

Por ejemplo:

```text
Modelo A:
Macro F1 = 0.42, 0.41, 0.40

Modelo B:
Macro F1 = 0.37, 0.49, 0.25
```

Aunque el modelo B tenga un fold mejor, el modelo A sería más estable.

---

## 11. Estructura estándar de las notebooks de modelos

Cada notebook posterior tendrá la misma estructura:

```text
1. Objetivo
2. Configuración
3. Carga de datos
4. Validación del dataset
5. Construcción del modelo
6. Entrenamiento por fold
7. Evaluación por fold
8. Agregación de resultados
9. Diagnósticos
10. Guardado
11. Observaciones
```

De esta manera podremos comparar modelos sin notebooks desordenadas o con procedimientos distintos.

---

## 12. Archivos de salida

La notebook `S07_00` debería guardar principalmente configuraciones y documentación:

```text
07_mnq_models/
│
├── config/
│   ├── stage_07_experimental_config.json
│   ├── stage_07_folds.csv
│   ├── stage_07_models.csv
│   └── stage_07_metrics.csv
│
├── sequences/
├── models/
├── predictions/
├── metrics/
└── reports/
```

El archivo principal sería:

```text
stage_07_experimental_config.json
```

Este archivo contendrá las reglas que deberán leer las notebooks siguientes.

---

# Resumen sencillo

La notebook `S07_00` responderá estas preguntas:

```text
¿Qué predecimos?
¿Con qué datos?
¿Qué historial verá el modelo?
¿Qué modelos probaremos?
¿Cómo se divide el tiempo?
¿Cómo entrenamos?
¿Cómo medimos el resultado?
¿Qué archivos se guardan?
```

El orden de desarrollo recomendado será:

```text
1. Objetivo y alcance
2. Punto de partida
3. Target OPC
4. Dataset principal
5. Ventanas
6. Walk-forward
7. Modelos
8. Métricas
9. Protocolo de entrenamiento
10. Configuración y guardado
```

Comenzaría ahora con el **Punto 1: Objetivo y alcance del Stage_07**, redactando la primera celda Markdown de la notebook.
